# Pole Figure Arithmetic

Two pole figures of the same specimen, one measured before a heat treatment and one
after. Subtract them and see what changed. It sounds like a one-line operation.

It is not, and the reason is worth understanding before reaching for the operator,
because it is the same reason that makes *every* quantitative comparison of two pole
figures harder than it looks:

1. **They do not share a support.** A `PoleFigure` holds *scattered* specimen
   directions — wherever the poles happened to land, or wherever the diffractometer
   happened to step. Two figures generally have no direction in common at all, so there
   is nothing to subtract pointwise.
2. **They do not share a scale.** One arrives in detector counts, one divided by its own
   maximum. Neither number means anything on its own, and the difference of two
   meaningless numbers is meaningless twice over.

So arithmetic is not a convenience that was left unwritten; it was **undefined** until
both problems were solved. This notebook works through the solution in the order the
dependencies force:

$$\text{shared support (resampling)} \;\longrightarrow\; \text{shared scale (m.r.d.)}
\;\longrightarrow\; \text{arithmetic}$$

and then shows what each operator actually produces, on simulated rolling textures where
the right answer is known in advance.

## 1. Setup: a copper specimen and two ideal components

Everything below is built from simulated textures rather than a data file, so that every
figure has a *known* correct answer to check against.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pytex import (
    STANDARD_FCC_ROLLING_COMPONENTS,
    CrystalPlane,
    FrameDomain,
    Handedness,
    Lattice,
    MillerIndex,
    OrientationSet,
    Phase,
    PoleFigure,
    ReferenceFrame,
    Rotation,
    S2Grid,
    SymmetrySpec,
    plot_pole_figure,
    plot_pole_figure_difference,
    raster_solid_angle_weights,
    spherical_angles_to_directions,
)
from pytex.texture import ODF, KernelSpec, PoleFigureResidualReport

crystal = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT)
specimen = ReferenceFrame("specimen", FrameDomain.SPECIMEN, ("RD", "TD", "ND"), Handedness.RIGHT)

copper = Phase(
    "copper-fcc",
    lattice=Lattice(3.615, 3.615, 3.615, 90.0, 90.0, 90.0, crystal_frame=crystal),
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=crystal),
    crystal_frame=crystal,
    space_group_symbol="Fm-3m",
)

pole_111 = CrystalPlane(MillerIndex((1, 1, 1), phase=copper), phase=copper)
pole_200 = CrystalPlane(MillerIndex((2, 0, 0), phase=copper), phase=copper)

COMPONENTS = {component.name: component for component in STANDARD_FCC_ROLLING_COMPONENTS}
for name, component in COMPONENTS.items():
    print(f"{name:8} {component.miller_label:14} Bunge {component.bunge_euler_deg}")

A real specimen is not a single orientation but a cloud of grains scattered around one.
`scatter_about` builds that cloud: it takes an ideal component, and rotates it by a
random axis through a random small angle, once per grain. The spread is the physically
meaningful parameter — a sharply rolled sheet might sit near 8 degrees, a weakly textured
one near 20.

In [ ]:
def scatter_about(name, *, spread_deg, count, rng):
    # A cloud of grain orientations scattered about an ideal component.
    ideal = COMPONENTS[name].orientation(
        specimen_frame=specimen,
        crystal_frame=crystal,
        symmetry=copper.symmetry,
        phase=copper,
    )
    axes = rng.normal(size=(count, 3))
    axes /= np.linalg.norm(axes, axis=1, keepdims=True)
    angles = np.abs(rng.normal(scale=np.deg2rad(spread_deg), size=count))
    perturbations = np.stack(
        [Rotation.from_axis_angle(axis, angle).as_matrix()
         for axis, angle in zip(axes, angles, strict=True)]
    )
    return OrientationSet.from_matrices(
        np.einsum("nij,jk->nik", perturbations, ideal.rotation.as_matrix()),
        crystal_frame=crystal,
        specimen_frame=specimen,
        symmetry=copper.symmetry,
        phase=copper,
    )


rng = np.random.default_rng(20260808)
copper_grains = scatter_about("copper", spread_deg=9.0, count=400, rng=rng)
cube_grains = scatter_about("cube", spread_deg=7.0, count=400, rng=rng)

copper_pf = PoleFigure.from_orientations(copper_grains, pole_111)
cube_pf = PoleFigure.from_orientations(cube_grains, pole_111)

print("copper {112}<111>:", copper_pf.sample_directions.shape[0], "poles")
print("cube   {001}<100>:", cube_pf.sample_directions.shape[0], "poles")
print("sampling reading:", copper_pf.sampling)

Note `sampling = "scattered_poles"`. That flag is not decoration — section 3 shows it
choosing between two different algorithms, and getting it wrong changes the answer.

400 grains gave 3200 poles because each orientation contributes the whole $\{111\}$
family: cubic symmetry makes 8 equivalent plane normals, and a measurement collects all
of them.

## 2. The problem: two figures share no support

Here is the obstacle stated numerically. For every pole of the copper figure, find the
closest pole of the cube figure.

In [ ]:
cosines = np.abs(copper_pf.sample_directions @ cube_pf.sample_directions.T)
nearest_deg = np.degrees(np.arccos(np.clip(cosines.max(axis=1), -1.0, 1.0)))

print(f"closest counterpart: min {nearest_deg.min():.3f} deg, "
      f"median {np.median(nearest_deg):.3f} deg")
print("directions the two figures share exactly:", int(np.sum(nearest_deg < 1e-9)))

Zero shared directions. Not "few" — none, and there is no reason there ever would be.
The supports are two unrelated point clouds on the sphere.

This is why `PoleFigure.__sub__` refuses rather than broadcasting: NumPy would happily
subtract two arrays of equal length and return numbers, and every one of them would
compare a pole in one place against an unrelated pole somewhere else.

In [ ]:
try:
    copper_pf - cube_pf
except ValueError as error:
    print(error)

## 3. Resampling: giving both figures the same support

The fix is to stop treating the figures as point clouds and evaluate both as *fields*, on
one grid chosen once.

### 3.1 The grid

`S2Grid.equispaced` places points on rings of constant polar angle, with the number of
points per ring scaled by $\sin\theta$ so the cells have nearly equal area. Each point
carries an integration weight, and the weights sum to 1 — those weights are what makes a
sum over the grid an approximation of an integral over the sphere.

In [ ]:
grid = S2Grid.equispaced(5.0, reference_frame=specimen, hemisphere="upper", antipodal=True)
print(f"{len(grid)} grid points, weights sum to {float(grid.weights.sum()):.12f}")

projected = grid.vectors.values
fig, ax = plt.subplots(figsize=(4.6, 4.6))
xy = np.column_stack([projected[:, 0], projected[:, 1]]) * np.sqrt(
    2.0 / (1.0 + np.abs(projected[:, 2]))
)[:, None]
ax.scatter(xy[:, 0], xy[:, 1], s=6, c=grid.weights * len(grid), cmap="viridis")
ax.set_aspect("equal")
ax.set_title(f"equal-area grid, {grid.resolution_deg:g} deg\ncolour = weight x N")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

The colouring is nearly uniform, which is the point of an *equal-area* grid: every point
speaks for about the same solid angle. Section 4 shows what happens with a grid that does
not have this property.

### 3.2 The kernel

Resampling asks: given values at scattered directions, what is the value *here*? The
answer is a weighted average of nearby data, with a kernel setting "nearby". PyTex uses
the von Mises-Fisher shape, written so it equals 1 at coincidence:

$$K(\hat{\mathbf{v}}, \hat{\mathbf{u}}) = \exp\!\big[\kappa\,(\hat{\mathbf{v}}\cdot\hat{\mathbf{u}} - 1)\big]$$

The single parameter $\kappa$ is fixed by the **halfwidth**, the angle at which the kernel
falls to half its peak. Setting $K = 1/2$ gives

$$\kappa = \frac{\ln 2}{1 - \cos(\text{halfwidth})}$$

The halfwidth is the consequential choice in this whole notebook. Too small and the result
reproduces sampling noise; too large and distinct texture components merge into one blob.
Set it from the angular resolution of the measurement — not by adjusting until the figure
looks nice.

In [ ]:
def kappa_from_halfwidth(halfwidth_deg):
    return np.log(2.0) / (1.0 - np.cos(np.deg2rad(halfwidth_deg)))


angle_deg = np.linspace(0.0, 45.0, 400)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for halfwidth in (5.0, 10.0, 20.0):
    kappa = kappa_from_halfwidth(halfwidth)
    ax.plot(angle_deg, np.exp(kappa * (np.cos(np.deg2rad(angle_deg)) - 1.0)),
            label=f"halfwidth {halfwidth:g} deg  (kappa = {kappa:.1f})")
    ax.plot([halfwidth], [0.5], "ko", markersize=5)
ax.axhline(0.5, color="grey", lw=0.8, ls="--")
ax.set_xlabel("angular separation (deg)")
ax.set_ylabel("kernel value")
ax.set_title("the S2 smoothing kernel; dots mark the halfwidth")
ax.legend()
plt.show()

### 3.3 Two estimators, and why the distinction is not pedantic

With a kernel in hand there are two ways to combine the neighbours, and **they are not
interchangeable**:

| | formula | correct for |
| --- | --- | --- |
| density estimation | $f(\hat{\mathbf{y}}) = \dfrac{\sum_i w_i K_i}{W\,\bar{K}}$ | a **cloud of poles**, where $w_i$ is one pole's weight |
| interpolation | $f(\hat{\mathbf{y}}) = \dfrac{\sum_i I_i K_i}{\sum_i K_i}$ | a **sampled field**, where $I_i$ is a density already measured at $\hat{\mathbf{u}}_i$ |

The first is a **sum**: more poles nearby means more density, which is exactly right when
each row of the figure *is* a pole. The second is a **weighted mean** (Nadaraya-Watson):
it reproduces the values it is given and does not care how densely they were sampled,
which is exactly right when each row is already a density.

`PoleFigure.sampling` records which reading applies, and `on_grid` picks the estimator
from it. Here is what using the wrong one costs. Take a field that is **exactly 1.0
everywhere**, sampled on a diffractometer-style raster that steps the tilt uniformly:

In [ ]:
raster_polar = np.repeat(np.arange(0.0, 90.1, 5.0), 72)
raster_azimuth = np.tile(np.arange(0.0, 360.0, 5.0), 19)
raster_directions = spherical_angles_to_directions(raster_polar, raster_azimuth).reshape(-1, 3)

flat_raster = PoleFigure(
    pole=pole_111,
    sample_directions=raster_directions,
    intensities=np.ones(raster_directions.shape[0]),   # exactly 1.0 everywhere
    specimen_frame=specimen,
    antipodal=True,
    sampling="sampled_density",
)

right = flat_raster.on_grid(grid, halfwidth_deg=10.0, normalize=False)
wrong = flat_raster.on_grid(grid, halfwidth_deg=10.0, estimator="density", normalize=False)

print(f"interpolate (correct): {right.intensities.min():.6f} to {right.intensities.max():.6f}")
print(f"density     (wrong)  : {wrong.intensities.min():.6f} to {wrong.intensities.max():.6f}")

In [ ]:
polar_deg = np.degrees(np.arccos(np.clip(np.abs(grid.vectors.values[:, 2]), -1.0, 1.0)))
order = np.argsort(polar_deg)

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(polar_deg[order], wrong.intensities[order], ".", ms=3, label="density estimator (wrong)")
ax.plot(polar_deg[order], right.intensities[order], ".", ms=3, label="interpolation (correct)")
ax.axhline(1.0, color="black", lw=0.9, ls="--", label="true value")
ax.set_xlabel("polar angle from ND (deg)")
ax.set_ylabel("resampled value")
ax.set_title("resampling a field that is 1.0 everywhere")
ax.legend()
plt.show()

The interpolating estimator returns 1.0 to twelve decimal places at every point, for any
halfwidth and any target direction. That is the *partition of unity*: a weighted mean of
identical values is that value, whatever the weights. It is the sharpest possible check
that this branch is a mean and not a sum.

The density estimator returns anything from 0.66 to 6.3 — a factor of nine — on data that
is constant. And the error is not noise, it is **structured**: it peaks at the pole,
because a uniform tilt raster crowds its points there. A sum counts that crowding as
density. This is the failure the `sampling` flag exists to prevent, and it is why the
reading is recorded on the object rather than guessed at the call site.

### 3.4 Resampling the simulated textures

With the right estimator chosen automatically, both textures land on the common grid.

In [ ]:
copper_grid = copper_pf.on_grid(grid, halfwidth_deg=10.0)
cube_grid = cube_pf.on_grid(grid, halfwidth_deg=10.0)

for label, figure in [("copper {112}<111>", copper_grid), ("cube {001}<100>", cube_grid)]:
    mean = float(np.sum(grid.weights * figure.intensities))
    print(f"{label:20} mean {mean:.9f} m.r.d.   peak {figure.intensities.max():5.2f} m.r.d.")

fig, axes = plt.subplots(1, 2, figsize=(11.0, 5.0))
plot_pole_figure(copper_grid, kind="scatter", title="copper component {111}", ax=axes[0])
plot_pole_figure(cube_grid, kind="scatter", title="cube component {111}", ax=axes[1])
plt.show()

Both means are 1.000000000 — not approximately, exactly, because `on_grid` normalizes by
default. That is the second half of the problem, and it is worth its own section.

## 4. Multiples of a random distribution

A pole figure is in **m.r.d.** when a random texture reads 1 everywhere. Then 2 means
twice as many poles point here as chance would put here, and the number means the same
thing in every figure ever measured.

The definition is a statement about an *integral*:

$$\frac{1}{4\pi}\oint P_{hkl}(\hat{\mathbf{y}})\,\mathrm{d}\Omega = 1
\qquad\Longleftrightarrow\qquad \sum_i w_i P_i = 1$$

with $w_i$ the solid-angle weights. Note what it is **not**: it is not the maximum, and it
is not the sum. Dividing by the maximum forces the peak to 1 whatever the texture
strength, which destroys the very quantity you wanted to measure.

### 4.1 Solid-angle weights for a measured raster

A grid from `S2Grid` carries its own weights. A diffractometer raster does not, and its
points are badly non-uniform — which is exactly the bias section 3.3 showed.
`raster_solid_angle_weights` supplies them: it groups points into rings and gives each
ring the solid angle of its band, $\cos\theta_{\text{lower}} - \cos\theta_{\text{upper}}$,
shared among its points.

In [ ]:
weights = raster_solid_angle_weights(raster_polar)
rings = np.arange(0.0, 90.1, 5.0)
per_ring = weights.reshape(rings.size, 72).sum(axis=1)

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.bar(rings, per_ring, width=3.5, label="solid-angle weight")
ax.axhline(1.0 / rings.size, color="crimson", lw=1.2, ls="--",
           label="what an unweighted average assumes")
ax.set_xlabel("ring polar angle (deg)")
ax.set_ylabel("share of the total weight")
ax.set_title("a uniform tilt raster is nowhere near uniform on the sphere")
ax.legend()
plt.show()

print(f"pole ring   {per_ring[0]:.5f}")
print(f"equator ring{per_ring[-1]:.5f}")
print(f"ratio       {per_ring[-1] / per_ring[0]:.1f}x")

The equatorial ring carries about 90 times the weight of the ring at the pole. An
unweighted average treats them as equals (the dashed line), which is how a naive mean over
raster data ends up telling you about the specimen normal and almost nothing else.

The weights are verifiable, not merely plausible. For $f = \cos\theta$ over a cap of
half-angle $t$ the solid-angle mean has the closed form $\sin^2 t\,/\,[2(1 - \cos t)]$:

In [ ]:
# The rings run 0 to 90 deg in 5 deg steps, and the outermost ring is given the
# half band it would have had if the raster continued -- so the region these
# weights describe is the cap out to 92.5 deg, not 90.
cap = np.deg2rad(90.0 + 5.0 / 2.0)
exact = float(np.sin(cap) ** 2 / (2.0 * (1.0 - np.cos(cap))))
weighted = float(np.sum(weights * np.cos(np.deg2rad(raster_polar))))
naive = float(np.mean(np.cos(np.deg2rad(raster_polar))))

print(f"analytic solid-angle mean : {exact:.6f}")
print(f"weighted average          : {weighted:.6f}   (error {abs(weighted - exact):.1e})")
print(f"unweighted average        : {naive:.6f}   (error {abs(naive - exact):.1e})")

The weighted average lands within $5\times10^{-4}$ of the closed form, and that error falls
like the square of the step. The unweighted average is off by 0.15 and does not converge to
this value however finely the raster is stepped: it is not a poor approximation of the
spherical mean, it is a different quantity.

### 4.2 Recovering a physical scale from arbitrary counts

Here is the practical consequence. Take the copper figure and multiply it by 8734, as if
it had been recorded with a different counting time. `normalize_to_mrd` divides by the
solid-angle-weighted mean and gives the original back.

In [ ]:
as_counts = PoleFigure(
    pole=pole_111,
    sample_directions=copper_grid.sample_directions,
    intensities=copper_grid.intensities * 8734.0,
    specimen_frame=specimen,
    antipodal=True,
    sampling="sampled_density",
)

print(f"mean of the raw figure : {as_counts.spherical_mean(integration_weights=grid.weights):.3f}")
restored = as_counts.normalize_to_mrd(integration_weights=grid.weights)
print(f"largest difference from the original after normalizing: "
      f"{float(np.max(np.abs(restored.intensities - copper_grid.intensities))):.3e}")

## 5. The operators

Both prerequisites are met: one grid, one scale. Now the arithmetic means something.

### 5.1 Addition — building a two-component texture

Pole densities add. A specimen that is 70% copper component and 30% cube produces the
weighted sum of their figures, and because both operands have mean 1, so does the blend.

In [ ]:
blend = 0.7 * copper_grid + 0.3 * cube_grid

print(f"0.7 * copper + 0.3 * cube : mean {float(np.sum(grid.weights * blend.intensities)):.9f}")
print(f"copper + cube             : mean "
      f"{float(np.sum(grid.weights * (copper_grid + cube_grid).intensities)):.9f}")

fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.0))
plot_pole_figure(copper_grid, title="copper", ax=axes[0])
plot_pole_figure(cube_grid, title="cube", ax=axes[1])
plot_pole_figure(blend, title="0.7 copper + 0.3 cube", ax=axes[2])
plt.show()

Exactly 1 and exactly 2. Scalar multiplication scales the density, so it scales the mean
with it — `0.7 * a + 0.3 * b` has mean $0.7 + 0.3 = 1$, and `a + b` has mean 2. The blend
figure shows both sets of maxima, at the strengths their volume fractions imply.

Multiplication is **scalars only**. The pointwise product of two pole densities is not a
pole density and has no accepted meaning, so it is not defined; for "how many times
stronger is this figure than that one" the operation you want is division.

### 5.2 Subtraction — and why it returns a different type

Subtracting the pure copper figure from the blend gives the change the cube addition made.

In [ ]:
residual = blend.difference(copper_grid, left_label="70/30 blend", right_label="pure copper")

print(type(residual).__name__)
print()
print(residual.describe())

Note the return type: **`PoleFigureDifference`, not `PoleFigure`**. This is deliberate.

`PoleFigure` enforces non-negative intensities, and it is right to — a pole density cannot
be negative. But a difference is *signed*, and its sign is its entire content: which
regions gained density and which lost it. Coercing the result into a non-negative type
would destroy exactly the information the subtraction was performed to obtain. So
subtraction returns its own type; addition, scaling and ratios stay non-negative and stay
`PoleFigure`.

The difference is drawn on a diverging colour scale with limits symmetric about zero, so
the neutral colour marks the zero crossing:

In [ ]:
plot_pole_figure_difference(residual)
plt.show()

print("integral of the difference:",
      f"{float(np.sum(grid.weights * residual.values)):.3e}")

Red where the blend has more $\{111\}$ density than pure copper, blue where it has less.
The cube component's poles are the red maxima; the blue regions are where the copper
component's own density was diluted to 70%.

The integral is zero to machine precision — necessarily, since both operands have mean 1
and the integral is linear. That is a useful check: a difference of two normalized figures
that does *not* integrate to zero means one of them was not normalized.

### 5.3 Deviation from random

Subtracting the scalar 1 from an m.r.d. figure gives the deviation from a random texture —
the natural way to see which directions are enriched and which depleted.

In [ ]:
deviation = blend - 1.0
print(f"range: {deviation.values.min():+.3f} to {deviation.values.max():+.3f} m.r.d.")
print(f"integrates to {float(np.sum(grid.weights * deviation.values)):.3e}")

plot_pole_figure_difference(deviation, title="blend, deviation from random")
plt.show()

### 5.4 Division — ratios, and where they stop meaning anything

The ratio of two figures answers "how many times stronger here than there". It is
non-negative, so it returns a `PoleFigure`.

In [ ]:
ratio = blend / copper_grid
print(f"ratio range: {ratio.intensities.min():.3f} to {ratio.intensities.max():.1f}")

well_supported = copper_grid.intensities > 0.05
print(f"restricted to where the denominator exceeds 0.05 m.r.d. "
      f"({well_supported.sum()} of {len(grid)} points): "
      f"{ratio.intensities[well_supported].min():.3f} to "
      f"{ratio.intensities[well_supported].max():.2f}")

That enormous upper figure is not a bug and not an interesting result — it is the ratio
evaluated where the *denominator* is nearly zero, which says nothing about the specimen and
everything about dividing by a small number. Restricting to directions where the
denominator is genuinely supported gives a range you can actually interpret.

Division by an exact zero refuses outright rather than substituting a value, because any
value substituted there would be an invention:

In [ ]:
holed = PoleFigure(
    pole=pole_111,
    sample_directions=copper_grid.sample_directions,
    intensities=np.where(np.arange(len(grid)) == 17, 0.0, 1.0),
    specimen_frame=specimen,
    antipodal=True,
    sampling="sampled_density",
)
try:
    blend / holed
except ValueError as error:
    print(error)

### 5.5 The guard rails

Every way two figures can differ while still *looking* combinable raises, with the reason
and — where relevant — the call that fixes it. These are the mistakes that would otherwise
produce a plausible, wrong, publishable number.

In [ ]:
copper_200 = PoleFigure.from_orientations(copper_grains, pole_200).on_grid(grid, halfwidth_deg=10.0)

for label, left, right in [
    ("different poles", copper_grid, copper_200),
    ("different support", copper_grid, cube_pf),
]:
    try:
        left - right
    except ValueError as error:
        print(f"[{label}]\n  {error}\n")

## 6. Transforming a figure

Three operations that change a figure's geometry rather than its values.

### 6.1 Rotation

`rotate` moves the poles and carries the densities with them — for bringing two specimens
measured in different settings into a common frame. The invariance worth checking: rotating
a *random* texture must leave it random.

### 6.2 Symmetrization

Rolled sheet is conventionally assumed orthorhombic about RD/TD/ND, and its pole figures
are averaged over that symmetry to suppress noise. That is an assumption about the
**process**, not a fact about the data, so PyTex never applies it implicitly and records it
on the result when you do.

The test below is the one that matters: take a deliberately asymmetric texture (the S
component), impose orthorhombic symmetry, and then rotate by 180 degrees about ND — an
operator of the group. If the symmetrization worked, the figure cannot change.

In [ ]:
orthorhombic = SymmetrySpec.from_point_group("222", reference_frame=specimen)
two_fold_nd = Rotation.from_axis_angle([0.0, 0.0, 1.0], np.pi)

s_pf = PoleFigure.from_orientations(scatter_about("s", spread_deg=8.0, count=300, rng=rng), pole_111)

plain = s_pf.on_grid(grid, halfwidth_deg=10.0)
turned = s_pf.rotate(two_fold_nd).on_grid(grid, halfwidth_deg=10.0)
symmetrized = s_pf.symmetrize(orthorhombic).on_grid(grid, halfwidth_deg=10.0)
symmetrized_turned = (
    s_pf.symmetrize(orthorhombic).rotate(two_fold_nd).on_grid(grid, halfwidth_deg=10.0)
)

print(f"as measured, after a 180 deg turn about ND: max change "
      f"{float(np.max(np.abs(turned.intensities - plain.intensities))):.4f} m.r.d.")
print(f"symmetrized, after the same turn          : max change "
      f"{float(np.max(np.abs(symmetrized_turned.intensities - symmetrized.intensities))):.2e} m.r.d.")
print()
print("sample symmetry recorded on the result:", symmetrized.sample_symmetry.point_group)

fig, axes = plt.subplots(1, 2, figsize=(11.0, 5.0))
plot_pole_figure(plain, title="S component, as simulated", ax=axes[0])
plot_pole_figure(symmetrized, title="after imposing 222 specimen symmetry", ax=axes[1])
plt.show()

The asymmetric figure changes by nearly 9 m.r.d. under the turn; the symmetrized one
changes by $10^{-15}$, which is machine precision. The symmetrization is exact, not
approximate.

### 6.3 Restricting the tilt range

An X-ray pole figure is only trustworthy out to the tilt where defocusing takes over,
typically 70-80 degrees. Beyond that the signal is instrument, not texture.
`restrict_polar_range` discards it explicitly, so the unreliable rim cannot quietly drive a
normalization or a residual.

In [ ]:
trimmed = copper_grid.restrict_polar_range(max_polar_deg=70.0)
print(f"{len(copper_grid.intensities)} points -> {len(trimmed.intensities)} points")
print(f"mean over the full hemisphere : "
      f"{float(np.sum(grid.weights * copper_grid.intensities)):.4f} m.r.d.")
print(f"mean over the measured cap    : {trimmed.spherical_mean():.4f} m.r.d.")

The two means differ, and that difference is a real caveat rather than a rounding error:
normalizing a partial pole figure puts its *measured cap* at mean 1, which equals the true
spherical mean only if the unmeasured cap has the same mean. PyTex states this rather than
hiding it — there is no way to know what is in the region you did not measure.

## 7. The payoff: residual pole figures for ODF quality

Everything above exists to make this section possible.

Inverting pole figures to an ODF produces a number for goodness of fit. That number says
*how badly* the ODF misses its own input data. It cannot say **where** — and where is what
identifies the cause. A miss concentrated in one region of the specimen sphere means an
unmodelled component or an uncorrected defocusing loss; noise spread evenly over the whole
figure means counting statistics. Same norm, completely different diagnosis, completely
different fix.

The residual pole figure is that diagnosis.

In [ ]:
odf = ODF.from_orientations(copper_grains, kernel=KernelSpec(halfwidth_deg=12.0))
measured_111 = PoleFigure.from_orientations(copper_grains, pole_111).on_grid(grid, halfwidth_deg=12.0)

report = PoleFigureResidualReport.from_odf(odf, measured_111)
print(report.describe())

In [ ]:
plot_pole_figure_difference(report.difference_figure())
plt.show()

A relative residual of about 1% and a residual figure that is small and unstructured: the
ODF reproduces the data it came from, which is the minimum any inversion must clear.

Now the instructive failure. Deliberately estimate the ODF with a kernel far too wide for
the texture — 30 degrees against a 9 degree spread — and look at the residual figure rather
than the number:

In [ ]:
over_smoothed = ODF.from_orientations(copper_grains, kernel=KernelSpec(halfwidth_deg=30.0))
bad = PoleFigureResidualReport.from_odf(over_smoothed, measured_111)

print(f"relative residual norm: {bad.relative_residual_norm:.4f}")
print(f"maximum absolute error: {bad.max_absolute_error:.3f} m.r.d.")

plot_pole_figure_difference(
    bad.difference_figure(), title="over-smoothed ODF: residual is structured, not noisy"
)
plt.show()

The residual is not scattered, it is **organized**: blue cores wrapped in red haloes. The
sign of the residual states that quantitatively rather than impressionistically. Split the
directions by how much density the measurement actually put there:

In [ ]:
values = bad.difference_figure().values
measured = measured_111.intensities
at_peaks = measured > np.percentile(measured, 95)
in_background = measured < np.percentile(measured, 40)

print(f"mean residual at the measured maxima : {values[at_peaks].mean():+.3f} m.r.d.")
print(f"mean residual where density is low   : {values[in_background].mean():+.3f} m.r.d.")

Strongly negative at the peaks, positive in the background: the model has smeared each peak
outward, so it **under**-predicts where the density truly concentrates and **over**-predicts
in the surrounding annulus. That is a kernel too wide, and nothing else looks quite like it.

A scalar error norm could not have told you this. It is the difference between "collect more
data" and "narrow the kernel", and only one of those is the right response.